In [35]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelBinarizer

In [36]:
# --- Caminhos para os arquivos ---
TRAIN_FILE_PATH = '../data/extracted_features/features.csv'
TEST_FILE_PATH = '../data/extracted_features/features_test.csv'

# --- Carregar os dados ---
try:
    train_df = pd.read_csv(TRAIN_FILE_PATH)
    test_df = pd.read_csv(TEST_FILE_PATH)
    print(f"Arquivos carregados com sucesso!")
    print(f"Formato dos dados de treino: {train_df.shape}")
    print(f"Formato dos dados de teste:  {test_df.shape}")
except FileNotFoundError:
    print(f"Erro: Arquivos não encontrados.")
    print(f"Verifique se os caminhos '{TRAIN_FILE_PATH}' e '{TEST_FILE_PATH}' estão corretos.")
    print("Se estiver no Google Colab, certifique-se de que os arquivos foram enviados e o caminho está correto.")

print("\n--- Amostra dos Dados de Treino ---")
display(train_df.head())

C:\Users\joaop\AppData\Local\Temp\ipykernel_13904\2960565307.py:7: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_FILE_PATH)


Arquivos carregados com sucesso!
Formato dos dados de treino: (367411, 66)
Formato dos dados de teste:  (99731, 66)

--- Amostra dos Dados de Treino ---


,nameseq,AAA,AAC,AAG,AAT,ACA,ACC,ACG,ACT,AGA,...,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT,label
0,ID_0_3_LTR,0.025262,0.01505,0.026337,0.019215,0.018678,0.011825,0.007659,0.01384,0.02365,...,0.01384,0.023246,0.026068,0.01935,0.020156,0.015318,0.016931,0.031846,0.026337,ALL_CLASSES
1,ID_1_5_LINE,0.022866,0.015244,0.019817,0.019817,0.021341,0.01372,0.006098,0.019817,0.015244,...,0.039634,0.02439,0.007622,0.016768,0.012195,0.015244,0.027439,0.02439,0.018293,ALL_CLASSES
2,ID_2_3_TIR,0.029851,0.014925,0.037313,0.022388,0.029851,0.029851,0.007463,0.007463,0.007463,...,0.022388,0.014925,0.007463,0.022388,0.0,0.007463,0.007463,0.007463,0.014925,ALL_CLASSES
3,ID_3_10_MITE,0.014493,0.014493,0.021739,0.014493,0.007246,0.0,0.0,0.036232,0.007246,...,0.014493,0.036232,0.014493,0.043478,0.014493,0.0,0.028986,0.065217,0.021739,ALL_CLASSES
4,ID_4_5_TIR,0.017321,0.011135,0.009898,0.018868,0.008042,0.015775,0.007733,0.017012,0.011135,...,0.017631,0.017631,0.016703,0.016084,0.020724,0.025363,0.015156,0.019487,0.030003,ALL_CLASSES


In [ ]:
# 1. Separar features (X) e labels (y)
# Primeiro, separamos os labels
y_train = train_df['label']
y_test = test_df['label']

# Em seguida, selecionamos apenas as colunas de features (k-mers)
# (Isso é mais seguro do que assumir que 'nameseq' e 'label' são as únicas colunas a remover)
feature_names = train_df.drop(columns=['nameseq', 'label']).columns
X_train = train_df[feature_names]
X_test = test_df[feature_names]

print(f"Número de features: {len(feature_names)}")
print(f"Exemplo de features: {feature_names[:5].to_list()}...")

# 2. FORÇAR A CONVERSÃO DE X PARA NUMÉRICO (A CORREÇÃO)
# O erro 'ValueError' indica que há strings ('AAA') nas colunas de features.
# 'errors='coerce'' transformará essas strings problemáticas em 'NaN' (Not a Number).
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')

# 3. Verificar se a coerção criou valores NaN (o que indica dados "sujos")
nan_in_train = X_train.isna().sum().sum()
if nan_in_train > 0:
    print(f"\nAlerta: {nan_in_train} valores não numéricos (strings) foram encontrados")
    print("nas colunas de features do treino e convertidos para NaN.")
    
    # Estratégia de Pré-processamento: Preencher NaNs com a média da coluna.
    # Isso permite que o StandardScaler funcione.
    X_train = X_train.fillna(X_train.mean())
    print("Valores NaN foram preenchidos com a média da sua respectiva coluna.")
else:
    print("\nColunas de features do treino parecem ser 100% numéricas.")

# 4. Aplicar o mesmo para o conjunto de teste
nan_in_test = X_test.isna().sum().sum()
if nan_in_test > 0:
    print(f"Alerta: {nan_in_test} valores não numéricos encontrados no teste.")
    # IMPORTANTE: Preencher NaNs do teste com a média do TREINO
    X_test = X_test.fillna(X_train.mean()) 
    print("Valores NaN do teste foram preenchidos com a média do TREINO.")

# 5. Aplicar LabelEncoder nos labels (y)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(f"\nClasses originais: {le.classes_}")
print(f"Classes codificadas: {np.unique(y_train_encoded)}")

Número de features: 64

Classes originais: ['ALL_CLASSES' 'label']
Classes codificadas: [0 1]


In [37]:

# 3. Aplicar StandardScaler nas features (X)
scaler = StandardScaler()

# Ajustar o scaler com os dados de TREINO
X_train_scaled = scaler.fit_transform(X_train)

# Apenas transformar os dados de TESTE (usando o ajuste do treino)
X_test_scaled = scaler.transform(X_test)

print(f"Shape dos dados de treino escalados: {X_train_scaled.shape}")

Shape dos dados de treino escalados: (367411, 64)


## Tarefa 4: Construção do Modelo (Classificador)

**Escolha do Classificador:** `RandomForestClassifier` (Floresta Aleatória)

**Justificativa:**
Para esta tarefa de classificação multiclasse com dados tabulares (onde as features são numéricas e representam frequências), o `RandomForestClassifier` é uma escolha robusta e de alto desempenho. A justificativa se baseia em:

* **Robustez a Overfitting:** Por ser um modelo de *ensemble* (baseado em *bagging*), ele combina múltiplas árvores de decisão, cada uma treinada em uma subamostra dos dados. Isso o torna significativamente menos propenso a sobreajuste (overfitting) do que uma única árvore de decisão.
* **Desempenho "Out-of-the-Box":** O RandomForest geralmente funciona muito bem sem a necessidade de um ajuste fino extensivo de hiperparâmetros, tornando-o um ótimo ponto de partida.
* **Interpretabilidade:** O modelo permite extrair a importância de cada feature (`feature_importances_`). No contexto do nosso problema, isso nos permitirá discutir quais k-mers (ex: 'AAA', 'CGC') foram mais decisivos para o modelo diferenciar as classes de TEs, o que é excelente para a seção de "Discussão" do relatório.
* **Não-linearidade:** Ele é capaz de capturar relações complexas e não-lineares entre as features, o que é provável que exista em dados biológicos.

In [38]:


print("Iniciando o treinamento do RandomForest...")

# Inicializar o classificador
# n_estimators=100 : Usar 100 árvores de decisão
# random_state=42 : Para garantir que o experimento seja reprodutível
# n_jobs=-1 : Usar todos os cores de CPU disponíveis para treinar mais rápido
clf = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)

# Treinar o modelo usando os dados de TREINO escalados
clf.fit(X_train_scaled, y_train_encoded)

print("Treinamento concluído.")

Iniciando o treinamento do RandomForest...
Treinamento concluído.


## Tarefa 5: Avaliação de Desempenho

Vamos avaliar o modelo no conjunto de **teste**.

Primeiro, mostraremos as métricas padrão (Acurácia e Relatório de Classificação) para uma visão geral.

Em seguida, calcularemos a **Área abaixo da curva Precisão-Revocação (AU-PRC)**, conforme solicitado na descrição do trabalho. Para problemas multiclasse, isso é tipicamente feito usando uma abordagem *One-vs-Rest* (OvR) e calculando uma média (ex: "micro" ou "macro").

In [39]:
# Fazer previsões no conjunto de TESTE
y_pred_test = clf.predict(X_test_scaled)

print("--- Avaliação no Conjunto de Teste ---")

# 1. Acurácia
acc = accuracy_score(y_test_encoded, y_pred_test)
print(f"Acurácia Total: {acc:.4f}\n")

# Vamos descobrir quais labels (classes) estão REALMENTE presentes no y_test
labels_presentes_no_teste = np.unique(y_test_encoded)
# Vamos pegar os nomes dessas classes
nomes_das_classes_presentes = le.inverse_transform(labels_presentes_no_teste)

print(f"Alerta: Apenas {len(labels_presentes_no_teste)} classe(s) foram encontradas no conjunto de teste: {nomes_das_classes_presentes}")
print("Isso indica que o arquivo 'features_test.csv' não é uma amostra representativa.")


# 2. Relatório de Classificação (com Precisão, Revocação, F1-Score por classe)
print("\nRelatório de Classificação (apenas para classes no conjunto de teste):")
# Usamos os parâmetros 'labels' e 'target_names' para refletir a realidade dos dados de teste
print(classification_report(
    y_test_encoded, 
    y_pred_test, 
    labels=labels_presentes_no_teste, 
    target_names=nomes_das_classes_presentes,
    zero_division=0 # Evita warnings se uma classe nunca for prevista (o que não deve acontecer aqui)
))

# Bônus: Verificar Acurácia no Treino (para checar overfitting)
y_pred_train = clf.predict(X_train_scaled)
acc_train = accuracy_score(y_train_encoded, y_pred_train)
print(f"\nAcurácia no Treino: {acc_train:.4f} (comparar com a acurácia de teste)")

--- Avaliação no Conjunto de Teste ---
Acurácia Total: 1.0000

Alerta: Apenas 1 classe(s) foram encontradas no conjunto de teste: ['ALL_CLASSES']
Isso indica que o arquivo 'features_test.csv' não é uma amostra representativa.

Relatório de Classificação (apenas para classes no conjunto de teste):
              precision    recall  f1-score   support

 ALL_CLASSES       1.00      1.00      1.00     99731

    accuracy                           1.00     99731
   macro avg       1.00      1.00      1.00     99731
weighted avg       1.00      1.00      1.00     99731


Acurácia no Treino: 1.0000 (comparar com a acurácia de teste)


In [25]:
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelBinarizer
import numpy as np

# 1. Obter as probabilidades de predição para o conjunto de teste
y_pred_proba = clf.predict_proba(X_test_scaled)

# 2. Verificar quantas classes o LabelEncoder aprendeu
n_classes = len(le.classes_)

# 3. Calcular a AU-PRC
if n_classes == 2:
    # --- CASO BINÁRIO (n_classes = 2) ---
    print("--- Detectado problema de classificação binária (2 classes) ---")
    
    # Para classificação binária, a função espera:
    # y_true: O array 1D original (e.g., [0, 0, 0, 1, 0])
    # y_score: O array 1D de probabilidades para a classe POSITIVA (classe '1')
    
    y_true_1d = y_test_encoded
    y_score_1d = y_pred_proba[:, 1] # Pega apenas a segunda coluna (probabilidades da classe 1)

    # Calcular a AU-PRC. Para binário, 'micro', 'macro' e 'weighted' são idênticos.
    au_prc = average_precision_score(y_true_1d, y_score_1d)
    
    print(f"\nÁrea sob a Curva PR (AU-PRC) no Teste: {au_prc:.4f}")
    
    # Adicionando contexto, já que seu teste só tem uma classe
    classes_no_teste = np.unique(y_test_encoded)
    if len(classes_no_teste) == 1:
        print(f"Nota: Seu conjunto de teste contém apenas a classe '{le.classes_[classes_no_teste[0]]}'.")
        print(f"A métrica acima é a AU-PRC calculada para a classe '{le.classes_[1]}'.")

else:
    # --- CASO MULTICLASSE (n_classes > 2) ---
    print(f"--- Detectado problema de classificação multiclasse ({n_classes} classes) ---")
    
    # Aqui, o LabelBinarizer é necessário e retornará um array 2D
    lb = LabelBinarizer()
    # Ajustar com todos os labels possíveis (do treino)
    lb.fit(y_train_encoded) 
    y_test_binarized = lb.transform(y_test_encoded)
    
    # Média "Micro"
    au_prc_micro = average_precision_score(y_test_binarized, y_pred_proba, average="micro")
    print(f"Área sob a Curva PR (AU-PRC) 'micro-average' no Teste: {au_prc_micro:.4f}")

    # Média "Macro"
    au_prc_macro = average_precision_score(y_test_binarized, y_pred_proba, average="macro")
    print(f"Área sob a Curva PR (AU-PRC) 'macro-average' no Teste: {au_prc_macro:.4f}")

    # --- AU-PRC por Classe (para o relatório) ---
    print("\n--- AU-PRC por Classe Individual ---")
    au_prc_per_class = average_precision_score(y_test_binarized, y_pred_proba, average=None)

    for i, class_name in enumerate(le.classes_):
        if i in np.unique(y_test_encoded):
             print(f"{class_name:>10}: {au_prc_per_class[i]:.4f}")
        else:
             print(f"{class_name:>10}: N/A (não estava no conjunto de teste)")

--- Detectado problema de classificação binária (2 classes) ---

Área sob a Curva PR (AU-PRC) no Teste: 0.0000
Nota: Seu conjunto de teste contém apenas a classe 'ALL_CLASSES'.
A métrica acima é a AU-PRC calculada para a classe 'label'.


c:\Users\joaop\OneDrive\Documentos\Ciencia_de_Dados\Quarto_Semestre\Bio Info\trabalho-TEsClassification\venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [ ]:
# 1. Obter as importâncias do modelo treinado
importances = clf.feature_importances_

# 2. Criar um DataFrame para facilitar a visualização
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
})

# 3. Ordenar o DataFrame pela importância
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

# 4. Selecionar o Top 20
top_20_features = feature_importance_df.head(20)

print("Top 20 Features Mais Importantes:\n")
print(top_20_features)

# 5. Plotar o gráfico
plt.figure(figsize=(10, 8))
plt.barh(top_20_features['feature'], top_20_features['importance'], color='skyblue')
plt.xlabel("Importância da Feature (Gini Importance)")
plt.ylabel("Feature (k-mer)")
plt.title("Top 20 Features Mais Importantes - RandomForest")
plt.gca().invert_yaxis() # Inverter o eixo Y para mostrar a mais importante no topo
plt.tight_layout()

# Salvar a figura (útil para o relatório)
plt.savefig("feature_importance_plot.png")

plt.show()